In [4]:
import numpy as np
from numpy.linalg import det, inv
from numpy import array

In [2]:
i_hat = array([1,0])
j_hat = array([2,2])

basis = array([i_hat, j_hat]).transpose()

determinant = det(basis)

print(basis)
print(determinant)

[[1 2]
 [0 2]]
2.0


In [5]:
A = array([
    [3,1,0],
    [2,4,1],
    [3,1,8]
])

B = array([54,12,6])

X = inv(A).dot(B)
print(X)

[19.8 -5.4 -6. ]


In [6]:
matrix = array([
    [2,6],
    [1,3]
])

rank = np.linalg.matrix_rank(matrix)
print(rank)

1


In [21]:
# logistic regression exercises
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import KFold, cross_val_score, train_test_split

In [19]:
df = pd.read_csv('https://bit.ly/3imidqa')

X = df.values[:, :-1]
y = df.values[:, -1]

kfold = KFold(n_splits=3, random_state=7, shuffle=True)
model = LogisticRegression(solver='liblinear')
results = cross_val_score(model, X, y, cv=kfold, error_score='raise')

In [20]:
print('accuracy mean: %.3f (stdev=%.3f)' % (results.mean(), results.std()))

accuracy mean: 0.979 (stdev=0.004)


In [22]:
# confusion matrix

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=7)
model.fit(X_train, y_train)
pred = model.predict(X_test)

cm = confusion_matrix(y_true=y_test, y_pred=pred)
print(cm)

[[165  12]
 [  0 267]]


In [ ]:
# test a prediction

while True:
    n = input('Input a color {red}, {green}, {blue}: ')
    (r,g,b)  = n.split(',')
    x = model.predict(np.array([[int(r), int(g), int(b)]]))

    if model.predict(np.array([[int(r), int(g), int(b)]])) == 0.0:
        print('Light')
    else:
        print('Dark')

Input a color {red}, {green}, {blue}:  0,0,0


Light


Input a color {red}, {green}, {blue}:  5,200,9


Dark


Input a color {red}, {green}, {blue}:  0,0,1


Light


In [ ]:
def rolling_origin_evaluation(
    y,
    model_type,
    model_params,
    exog=None,
    start_train_size=156,
    horizons=(1, 13, 26, 52),
    step=13,
    sp=52,
    checkpoint_path=None,
    start_params=None,
    random_state=None,
    verbose=False
):
    '''
    Rolling-origin evaluation using expanding window (point forecasts).
    Reports on mean forecast performance via RMSE and MAE
    Returns one record per (origin, horizon).
    
    Parameters
    ----------
    y : pd.Series or pd.DataFrame
        The target time series data with a proper index (DatetimeIndex preferred).
    model_type : str
        Identifier for the model architecture to be fitted.
    model_params : dict
        Configuration parameters passed to the model fitting function.
    exog : pd.Series or pd.DataFrame, optional
        Exogenous variables aligned with `y`.
    start_train_size : int, default 156
        Initial number of observations used for the first training window.
    horizons : tuple of int, default (1, 13, 26, 52)
        Specific forecast steps (lead times) to evaluate at each origin.
    step : int, default 13
        The number of periods to advance the origin between folds.
    sp : int, default 52
        Seasonal period used for calculating Seasonal Naive benchmarks.
    checkpoint_path : str, optional
        Path to a .pkl file for saving/loading progress to handle interruptions.
    random_state : int, optional
        Seed for reproducibility in stochastic models.
    verbose : bool, default False
        If True, prints debug information and detailed error logs.

    Returns
    -------
    pd.DataFrame
        A detailed log containing one row per (origin, horizon) pair with 
        true values, predictions, and calculated error metrics (RMSE, MAE).
        
    Notes
    -----
    The function utilizes a generator to yield `y_train` and `y_test` for each 
    fold. `y_train` grows in size by `step` observations each iteration, 
    implementing the expanding window strategy.
    '''
    start_time = time.time()
    
    records = []
    H_MAX = max(horizons)
    
    completed_origins = set()  
    if checkpoint_path and os.path.exists(checkpoint_path):
        with open(checkpoint_path, 'rb') as f:
            ckpt = pickle.load(f)
            records = ckpt['records']
            completed_origins = ckpt['completed_origins']
            
        if verbose:
            print(f'Loaded checkpoint with {len(completed_origins)} completed folds')
    
    origins = range(start_train_size, len(y) - H_MAX + 1, step)    # expanding window 
    n_folds_total = len(origins) 
    
    # initialize the rolling_origins() generator 
    gen = rolling_origins(y, exog, start_train_size, H_MAX, step)
    
    pbar = tqdm(gen, total=n_folds_total, desc='Rolling-origin folds', unit='fold')
    
    completed_count = len(completed_origins)

    for t, y_train, y_test, exog_train, exog_test in pbar:
        
        if t in completed_origins:
            continue

        try:
            fitted = fit_mean_model(
                y_train,
                model_type,
                model_params,
                exog=exog_train,
                start_params=start_params,
                verbose=verbose
            )

            start_params = fitted.get('start_params')
            
            fc = forecast_mean_model(
                fitted,
                horizon=H_MAX,
                exog=exog_test
            )
            
            mean = fc['mean']

            # Check for NaN or implausible values across all horizons.  Validate forecasts
            if mean.isnull().any() or fc['sigma'].isnull().any():
                tqdm.write(f"Skipping fold t={t}: NaN in forecast")
                continue
            
            if (mean < 1500).any() or (mean > 2500).any():
                tqdm.write(f"Skipping fold t={t}: implausible forecast range [{mean.min():.1f}, {mean.max():.1f}]")
                continue

            # loop over horizons
            for h in horizons:
                y_true = y_test.iloc[h - 1]
                y_pred = mean.iloc[h - 1]

                # seasonal naive
                naive_fcst = seasonal_naive_forecast(
                    y_train=y_train.values,
                    horizon=h,
                    sp=sp
                )
                y_naive = naive_fcst[h - 1]

                err = y_pred - y_true
                naive_err = y_naive - y_true

                # debug – first successful error only
                if verbose and len(records) == 0:
                    print('DEBUG err value:', err)
                    print('DEBUG err type:', type(err))
                    print('DEBUG err ndim:', np.ndim(err))

                records.append({
                    'origin'     : y.index[t],
                    'origin_idx' : t,
                    'horizon'    : h,
                    'model'      : model_type,
                    'y_true'     : y_true,
                    'y_pred'     : y_pred,
                    'y_naive'    : y_naive,
                    'error'      : err,
                    'naive_error': naive_err,
                    'abs_error'  : abs(err),
                    'sq_error'   : err**2,
                    'train_size' : len(y_train)
                })
                
            completed_origins.add(t)
            completed_count += 1
            
            if checkpoint_path:
                with open(checkpoint_path, 'wb') as f:
                    pickle.dump(
                        {
                            'completed_origins': completed_origins,
                            'records'          : records
                        },
                        f
                    )
                
            end_time = time.time()
            elapsed_time = end_time - start_time
            avg_per_fold = elapsed_time / completed_count
            eta = avg_per_fold * (n_folds_total - completed_count)
                
            pbar.set_postfix(
                elapsed_min=f'{elapsed_time/60:.2f}',
                eta_min=f'{eta/60:.2f}'
            )

        except Exception as e:
            tqdm.write(f'Fold failed at t={t}: {e}')
            if verbose:
                traceback.print_exc()                         # prints full traceback to stderr
                # logger.exception(f"Fold failed at t={t}")   # alternative approach to debugging
            continue

    return pd.DataFrame(records)